[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/12_embedders_modernos/12_proyecto_embedders_modernos.ipynb)

# Proyecto 12 · Comparativa de embedders para Políglota: calidad vs memoria vs latencia

**Nivel:** 🟠 Avanzado · **Duración:** ≈ 2 h · **GPU:** T4 recomendada para los 4–6 modelos (CPU basta con `FAST_DEV_RUN = True` y 2 modelos) · **Cómputo:** ≈ 1–2 unidades de Colab en T4 con `FAST_DEV_RUN = False`.

**Conexión.** *Viene de:* módulo 11 (tu embedder contrastivo ES/EN: lo medirás contra estos baselines si lo tienes guardado) y de la lección 12. *Va hacia:* módulo 13 (el mejor embedder será el retriever denso del sistema híbrido) y 14 (la configuración elegida decide el tamaño del índice).

## 1 · Contexto de negocio
Eres ML engineer en **Políglota Labs**. Vamos a indexar **10 millones de fragmentos** de documentación (ES/EN/código). El CTO te da un límite: **el índice de vectores debe caber en ≈ 20 GB de RAM** y la búsqueda debe ser rápida. Tu tarea: construir una **comparativa reproducible de 4–6 embedders**, cada uno con **truncado Matryoshka** y **cuantización** (`float32`, `int8`, binaria, binaria+rescoring), y entregar una **recomendación justificada** con la frontera de Pareto calidad-memoria-latencia.

## 2 · Dataset
Corpus Políglota vía `carga_corpus()` (Wikipedia ES/EN + CodeSearchNet si hay acceso a Hugging Face; si no, el **corpus sintético** del módulo 00: texto ES/EN de plantillas + funciones de la stdlib como código — verás un `⚠️`). Con el corpus sintético los números **no son comparables** con los del real, pero el pipeline y las conclusiones relativas (int8 ≈ float32, binaria necesita rescoring…) sí se ejercitan.

## 3 · Entregables
1. `construye_store` / `buscar` funcionando para 4 precisiones (`float32`, `int8`, `binario`, `binario+rescore`) y 4 escalas de dimensión (D, D/2, D/4, D/8).
2. Una **tabla de resultados** (`RESULTS`) con: modelo, dimensión, precisión, nDCG@10, overlap@10 vs float32, bytes/vector, MB para 10 M docs, ms/query.
3. La **frontera de Pareto** (memoria vs nDCG@10) y una **recomendación** para el presupuesto de 20 GB.
4. Una conclusión escrita (≤ 10 líneas) que cite números de **tu** tabla.

## 4 · Rúbrica (umbrales calibrados con lo que MIDE la solución de referencia)
Los umbrales de abajo son **invariantes relativas** a la propia ejecución (no valores absolutos), porque el nDCG absoluto depende del corpus y de los modelos que puedas descargar. La solución de referencia se ejecutó en un entorno **sin acceso a Hugging Face** (corpus sintético + embedders de respaldo TF-IDF+SVD + un modelo local diminuto con pesos aleatorios para ejercitar la ruta de `sentence-transformers`); los valores medidos están en la última columna. **No se verificó con modelos reales descargados**; si en Colab tus números reales son mejores, mejor.

| # | Criterio | Umbral | Medido por la solución de referencia |
|---|---|---|---|
| R1 | `trunc_renorm` devuelve vectores de norma 1 | `max |‖x‖−1| < 1e-5` | 1.2e-7 (precisión de float32) |
| R2 | Binario idéntico a `sentence-transformers` (`ubinary`) e `int8` a ≤ 1 nivel | exacto / ≤ 1 | binario idéntico; int8 ≤ 1 nivel |
| R3 | `int8` conserva la calidad: nDCG@10 ≥ float32 − 0.03 en **todas** las filas | cumplido en el 100 % de filas | peor caso −0.008 de nDCG@10 |
| R4 | `int8` conserva el ranking: overlap@10 (vs float32 a igual dimensión) medio ≥ 0.85 | ≥ 0.85 | media 0.948 con los respaldos TF-IDF+SVD y 0.895 con el modelo diminuto aleatorio (mínimos 0.865 y 0.811) |
| R5 | El rescoring ayuda: overlap(binario+rescore) ≥ overlap(binario) en ≥ 90 % de las filas | ≥ 90 % | 100 % de las filas |
| R6 | Memoria: `int8` = m bytes/vec y binario = m/8 bytes/vec | exacto | exacto |
| R7 | La frontera de Pareto tiene ≥ 3 puntos y ≥ 1 punto que **no** es `float32` | ≥ 3 | 8 puntos (respaldos) / 5 puntos (modelo diminuto) |
| R8 | La recomendación respeta el presupuesto de 20 GB (10 M docs) | ≤ 20 GB | 3.2 GB (respaldos) / 0.64 GB (modelo diminuto) |
| R9 | Conclusión escrita con ≥ 3 números de tu tabla | — | revisión manual |

## 5 · Preparación (celdas dadas, no hay que modificarlas)

In [ ]:
!pip install -q -U sentence-transformers transformers scikit-learn matplotlib pandas mteb

In [ ]:
import os, sys, time, math, random, json, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True      # True: tamaños diminutos (CPU en minutos). Pon False en Colab con GPU para la escala completa.
N_THREADS = int(os.environ.get("N_THREADS", "1" if FAST_DEV_RUN else "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))

try:
    import torch
    torch.manual_seed(SEED)
    torch.set_num_threads(N_THREADS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:      # los fragmentos "desde cero" de NumPy funcionan sin torch
    torch, device = None, "cpu"
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · threads={N_THREADS}")

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
def ndcgs_por_query(run: dict, qrels: dict, k: int = 10):
    """nDCG@k de cada query (mismo filtro que evaluate(): solo queries con algún relevante). Sirve para bootstrap."""
    return np.array([ndcg_at_k(run.get(q, []), r, k) for q, r in qrels.items() if any(v > 0 for v in r.values())])

In [ ]:
# Catálogo de embedders multilingües "pequeños" (caben en CPU/T4). Prefijos copiados de las tarjetas de modelo:
# VERIFÍCALOS en la tarjeta antes de usar un modelo en producción (cambian entre versiones).
MODELOS = {
    "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small", q="query: ", d="passage: "),
    "multilingual-e5-base": dict(hf="intfloat/multilingual-e5-base", q="query: ", d="passage: "),
    "paraphrase-multilingual-MiniLM-L12-v2": dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", q="", d=""),
    "gte-multilingual-base": dict(hf="Alibaba-NLP/gte-multilingual-base", q="", d="", trust=True),
    "bge-m3": dict(hf="BAAI/bge-m3", q="", d=""),
    "nomic-embed-text-v2-moe": dict(hf="nomic-ai/nomic-embed-text-v2-moe", q="search_query: ", d="search_document: ", trust=True),
    "Qwen3-Embedding-0.6B": dict(hf="Qwen/Qwen3-Embedding-0.6B", d="",
                                 q="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"),
    "embeddinggemma-300m": dict(hf="google/embeddinggemma-300m", q="task: search result | query: ", d="title: none | text: ", gated=True),
}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

class HashSVDEmbedder:
    """Fallback SIN descargas: TF-IDF de n-gramas de caracteres + SVD. NO es multilingüe ni semántico:
    sirve para ejercitar el pipeline (truncado, cuantización, métricas) cuando HF no es accesible."""
    def __init__(self, dim: int = 64, ngram=(3, 5), analyzer: str = "char_wb"):
        self.dim = dim
        self.tfidf = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, sublinear_tf=True, min_df=1)
        self.svd = TruncatedSVD(n_components=dim, random_state=SEED)
    def fit(self, textos):
        X = self.tfidf.fit_transform(textos)
        self.svd.n_components = min(self.dim, X.shape[1] - 1, X.shape[0] - 1)
        self.svd.fit(X)
        return self
    def encode(self, textos):
        Z = self.svd.transform(self.tfidf.transform(textos))
        return (Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9)).astype(np.float32)

class Embedder:
    """Interfaz común: encode_queries / encode_docs (aplican el prefijo de cada modelo) -> float32 normalizado."""
    def __init__(self, clave, cfg, model=None, fallback=None):
        self.clave, self.cfg, self.model, self.fallback = clave, cfg, model, fallback
        self.is_real = model is not None
    def _enc(self, textos, prefijo, bs=16):
        if self.model is not None:
            E = self.model.encode([prefijo + t for t in textos], batch_size=bs, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False)
            return E.astype(np.float32)
        return self.fallback.encode([prefijo + t for t in textos])
    def encode_queries(self, textos): return self._enc(textos, self.cfg.get("q", ""))
    def encode_docs(self, textos): return self._enc(textos, self.cfg.get("d", ""))
    @property
    def n_params(self):
        return sum(p.numel() for p in self.model.parameters()) if self.model is not None else 0

def cargar_embedder(clave, textos_ajuste=None, max_len=256):
    """Carga con sentence-transformers; si no se puede (sin red, modelo con licencia, falta de RAM) cae al fallback."""
    cfg = MODELOS[clave]
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(cfg["hf"], device=device, trust_remote_code=cfg.get("trust", False))
        m.max_seq_length = min(m.max_seq_length or max_len, max_len)
        return Embedder(clave, cfg, model=m)
    except Exception as e:
        print(f"[embedder] {clave}: no se pudo cargar ({type(e).__name__}: {str(e)[:70]})")
        if textos_ajuste is None:
            return None
        print(f"[embedder] {clave}: usando fallback TF-IDF+SVD (NO representa al modelo real)")
        return Embedder(clave + " (fallback)", cfg, fallback=HashSVDEmbedder().fit(textos_ajuste))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

CORPUS = carga_corpus(FAST_DEV_RUN)
DOC_IDS = [d["id"] for d in CORPUS["docs"]]
DOC_TXT = [d["text"] for d in CORPUS["docs"]]
QRYS, QRELS = CORPUS["queries"], CORPUS["qrels"]
Q_TXT = [q["text"] for q in QRYS]

POPCNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def topk_scores(S, k):
    """Índices de los k mayores scores por fila, ordenados de mayor a menor."""
    idx = np.argpartition(-S, min(k, S.shape[1] - 1), axis=1)[:, :k]
    orden = np.take_along_axis(S, idx, 1).argsort(1)[:, ::-1]
    return np.take_along_axis(idx, orden, 1)

def overlap_at_k(est, ref):
    return float(np.mean([len(set(a) & set(b)) / len(b) for a, b in zip(est, ref)]))

def aumenta(E, copias=40, sigma=0.5, seed=0):
    """Corpus ampliado con casi-duplicados (cos ≈ 0.9) para crear vecinos difíciles."""
    rng = np.random.default_rng(seed)
    bloques = [E]
    for _ in range(copias):
        ruido = rng.normal(size=E.shape)
        ruido *= sigma / np.linalg.norm(ruido, axis=1, keepdims=True)
        X = E + ruido
        bloques.append((X / np.linalg.norm(X, axis=1, keepdims=True)).astype(np.float32))
    return np.vstack(bloques)

def run_desde_indices(idx):
    return {q["qid"]: [DOC_IDS[j] for j in idx[i]] for i, q in enumerate(QRYS)}

N_DOCS_PROD = 10_000_000          # tamaño de producción para extrapolar memoria
PRESUPUESTO_GB = 20               # límite del CTO
print(CORPUS["source"], len(DOC_IDS), "docs ·", len(QRYS), "queries")

## 6 · Paso 1 — Codificar con 4–6 embedders (TODO 1)
Usa `cargar_embedder(clave)` (ya definido): devuelve un `Embedder` con `encode_queries` / `encode_docs` que **aplican el prefijo correcto**. Si no se puede descargar, devuelve `None`. En ese caso el proyecto añade **embedders de respaldo** (TF-IDF+SVD con distintas configuraciones) para que la comparativa siga ejecutándose.

Mide para cada modelo: embeddings de queries/documentos, **ms por documento** y **ms por query**, y el número de parámetros.

<details><summary>🪜 Pista 1 (idea)</summary>
Itera sobre una lista de claves de <code>MODELOS</code>; por cada una llama a <code>cargar_embedder</code>, mide el tiempo con <code>time.perf_counter()</code> alrededor de <code>encode_docs</code> y <code>encode_queries</code>, y guarda todo en un diccionario <code>BANCO[clave]</code>.
</details>
<details><summary>🪜 Pista 2 (casi la API)</summary>
<code>t0 = time.perf_counter(); Ed = e.encode_docs(DOC_TXT); ms_doc = (time.perf_counter()-t0)/len(DOC_TXT)*1e3</code> · guarda <code>dict(Eq=Eq, Ed=Ed, ms_doc=..., ms_q=..., params_M=e.n_params/1e6, real=e.is_real)</code>.
</details>

In [ ]:
CLAVES_REALES = (["multilingual-e5-small", "paraphrase-multilingual-MiniLM-L12-v2"] if FAST_DEV_RUN else
                 ["multilingual-e5-small", "multilingual-e5-base", "gte-multilingual-base",
                  "Qwen3-Embedding-0.6B", "bge-m3", "embeddinggemma-300m"])
FALLBACKS = {   # respaldo si no hay red: (dim, n-gramas, analizador)
    "tfidf-svd-64 (fallback)": (64, (3, 5), "char_wb"),
    "tfidf-svd-32 (fallback)": (32, (2, 4), "char_wb"),
    "tfidf-svd-80 (fallback)": (80, (3, 5), "char_wb"),
    "palabras-svd-48 (fallback)": (48, (1, 2), "word"),
}

def codifica_modelo(clave, e):
    """TODO 1: devuelve dict(Eq, Ed, ms_doc, ms_q, params_M, real) para el embedder `e`."""
    # TODO 1.a: mide el tiempo de encode_docs(DOC_TXT) y encode_queries(Q_TXT)
    # TODO 1.b: devuelve el diccionario descrito arriba
    raise NotImplementedError("TODO 1")

def construye_banco():
    banco = {}
    for clave in CLAVES_REALES:
        e = cargar_embedder(clave)
        if e is not None:
            banco[clave] = codifica_modelo(clave, e)
    if len(banco) < 4:                                    # completa hasta 4 con respaldos (solo sin red)
        for nombre, (dim, ng, an) in FALLBACKS.items():
            if len(banco) >= 4:
                break
            fb = Embedder(nombre, {}, fallback=HashSVDEmbedder(dim, ng, an).fit(DOC_TXT))
            banco[nombre] = codifica_modelo(nombre, fb)
    return banco

try:
    BANCO = construye_banco()
    print({k: (v["Ed"].shape, round(v["ms_doc"], 2)) for k, v in BANCO.items()})
except NotImplementedError as e:
    BANCO = {}
    print("⏳ pendiente:", e)

## 7 · Paso 2 — Truncado Matryoshka (TODO 2)
Implementa `trunc_renorm(E, m)`: conserva las primeras `m` dimensiones y **renormaliza** cada fila a norma 1 (protege la división por cero con `np.maximum(norma, 1e-9)`).
Después `dims_grid(D)` devuelve las 4 escalas `D, D/2, D/4, D/8` redondeadas a múltiplos de 8 (la binaria empaqueta de 8 en 8) y nunca menores que 8.

In [ ]:
def trunc_renorm(E, m):
    """Primeras m dimensiones + renormalización L2."""
    # TODO 2: implementa el truncado y la renormalización
    raise NotImplementedError("TODO 2")

def dims_grid(D):
    """D, D/2, D/4, D/8 redondeadas a múltiplos de 8, sin repetir y >= 8."""
    return sorted({max(8, (D // f) // 8 * 8) for f in (1, 2, 4, 8)}, reverse=True)

try:
    X = trunc_renorm(next(iter(BANCO.values()))["Ed"], 16) if BANCO else None
    if X is not None:
        print("normas:", np.linalg.norm(X, axis=1)[:3])
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## 8 · Paso 3 — Cuantización `int8` y binaria (TODO 3 y 4)
**TODO 3 (`int8`).** `ajusta_rangos` calcula `lo`/`hi` por dimensión **sobre el corpus**. `cuantiza_int8` aplica $q=\operatorname{round}((x-lo)/((hi-lo)/255))-128$ recortado a $[-128,127]$ (`np.int8`) y `descuantiza_int8` lo invierte.

**TODO 4 (binaria).** `cuantiza_binario` empaqueta `E > 0` con `np.packbits(..., axis=1)`. `hamming(qb, db)` devuelve la matriz (nº queries × nº docs) de distancias de Hamming usando XOR + la tabla `POPCNT` (procesa los documentos por bloques para no agotar RAM).

<details><summary>🪜 Pista 1 (idea)</summary>
Hamming = nº de bits distintos = popcount(a XOR b). Un byte tiene 256 valores posibles: <code>POPCNT[x]</code> te da el popcount por tabla. Suma sobre los bytes de cada vector.
</details>
<details><summary>🪜 Pista 2 (casi la API)</summary>
<code>x = qb[:, None, :] ^ db[None, s:s+bloque, :]</code> y después <code>POPCNT[x].sum(-1)</code>. Para int8: <code>paso = np.maximum(hi-lo, 1e-9)/255</code>; <code>np.clip(np.round((E-lo)/paso)-128, -128, 127).astype(np.int8)</code>.
</details>

In [ ]:
def ajusta_rangos(E_calib):
    # TODO 3.a: devuelve (lo, hi) por dimensión calculados sobre el corpus
    raise NotImplementedError("TODO 3.a")

def cuantiza_int8(E, lo, hi):
    # TODO 3.b: cuantización escalar a int8 con 256 niveles
    raise NotImplementedError("TODO 3.b")

def descuantiza_int8(Q, lo, hi):
    # TODO 3.c: inversa aproximada, devuelve float32
    raise NotImplementedError("TODO 3.c")

def cuantiza_binario(E, umbral=0.0):
    # TODO 4.a: bits (E > umbral) empaquetados en uint8, forma (N, D/8)
    raise NotImplementedError("TODO 4.a")

def hamming(qb, db, bloque=2048):
    # TODO 4.b: matriz (nq, nd) de distancias de Hamming (XOR + POPCNT), por bloques de documentos
    raise NotImplementedError("TODO 4.b")

try:
    ref_bin = np.packbits(np.random.default_rng(0).normal(size=(4, 32)) > 0, axis=1)
    print("hamming(x, x) = 0:", int(hamming(ref_bin, ref_bin).trace()) == 0)
except NotImplementedError as e:
    print("⏳ pendiente:", e)

## 9 · Paso 4 — Búsqueda con cada precisión (TODO 5)
`construye_store(Ed, m)` prepara, para una escala `m`, los vectores truncados en las tres representaciones; `buscar(Eq, store, precision, k, over)` devuelve los índices del top-k:

- `float32`: producto punto exacto sobre los vectores truncados y renormalizados.
- `int8`: la query en `float32` contra los documentos `int8` **descuantizados**.
- `binario`: ordena por menor Hamming entre `Q > 0` empaquetada y los bits de los documentos.
- `binario+rescore`: recupera `k*over` candidatos por Hamming y **reordena** con la query `float32` contra los `int8` descuantizados.

<details><summary>🪜 Pista 1 (idea)</summary>
El rescoring es "dos etapas": primero un top-(k·over) barato (Hamming), luego un top-k caro pero solo sobre esos candidatos (producto punto con vectores descuantizados).
</details>
<details><summary>🪜 Pista 2 (casi la API)</summary>
<code>H = hamming(np.packbits(Q &gt; 0, axis=1), store["bin"])</code> · <code>cand = topk_scores(-H.astype(np.float32), k*over)</code> · para cada query <code>v = descuantiza_int8(store["int8"][cand[i]], lo, hi)</code> y <code>cand[i][np.argsort(-(v @ Q[i]))[:k]]</code>.
</details>

In [ ]:
def construye_store(Ed, m):
    # TODO 5.a: trunca+renormaliza, ajusta rangos, cuantiza a int8 y a binario. Devuelve un dict con
    #           claves "m", "f32", "int8", "lo", "hi", "bin"
    raise NotImplementedError("TODO 5.a")

def buscar(Eq, store, precision, k=10, over=4):
    # TODO 5.b: implementa las 4 precisiones: "float32", "int8", "binario", "binario+rescore"
    raise NotImplementedError("TODO 5.b")

try:
    c0 = next(iter(BANCO.values()))
    st0 = construye_store(c0["Ed"], dims_grid(c0["Ed"].shape[1])[1])
    print({p: buscar(c0["Eq"], st0, p).shape for p in ("float32", "int8", "binario", "binario+rescore")})
except (NotImplementedError, StopIteration) as e:
    print("⏳ pendiente:", repr(e))

## 10 · Paso 5 — La rejilla completa (TODO 6)
Para cada modelo × escala × precisión calcula:
- `ndcg10` y `recall10` sobre el corpus original (`store` construido con `Ed`),
- `overlap10`: fracción del top-10 exacto en `float32` **a la misma dimensión m** que conserva el método (aísla el error de *cuantización* del de truncado), medido sobre el **corpus ampliado** `aumenta(Ed)` (vecinos difíciles),
- `bytes_vec` en RAM (`float32`: 4m · `int8`: m · `binario` y `binario+rescore`: m/8), `MB_10M` = bytes · 10 M / 10⁶ y `ms_query` (latencia media por query sobre el corpus ampliado).

<details><summary>🪜 Pista 1 (idea)</summary>
Construye dos <code>store</code> por (modelo, m): uno con <code>Ed</code> (para nDCG) y otro con <code>E_big</code> (para overlap y latencia). La referencia <code>ref_m</code> es el top-10 exacto <code>float32</code> a la misma dimensión <code>m</code> sobre <code>E_big</code>: se obtiene con <code>buscar(Eq, st_big, "float32")</code>.
</details>

In [ ]:
BYTES = {"float32": lambda m: 4 * m, "int8": lambda m: m, "binario": lambda m: m // 8, "binario+rescore": lambda m: m // 8}

def evalua_rejilla(banco):
    """TODO 6: devuelve un DataFrame con una fila por (modelo, dim, precision)."""
    filas = []
    # TODO 6.a: para cada modelo: E_big = aumenta(Ed)
    # TODO 6.b: para cada m en dims_grid(D): ref_m = buscar(Eq, st_big, "float32"); y para cada precision:
    #           ndcg10/recall10 (store con Ed), overlap10 vs ref_m y ms_query (store con E_big)
    # TODO 6.c: añade bytes_vec = BYTES[precision](m) y MB_10M = bytes_vec * N_DOCS_PROD / 1e6
    raise NotImplementedError("TODO 6")

try:
    RESULTS = evalua_rejilla(BANCO)
    display(RESULTS.round(3).head(12))
except NotImplementedError as e:
    RESULTS = None
    print("⏳ pendiente:", e)

## 11 · Paso 6 — Frontera de Pareto y recomendación (TODO 7)
Una configuración está en la **frontera de Pareto** si ninguna otra tiene a la vez **menos memoria y mejor (o igual) nDCG@10**. Implementa:
- `frontera_pareto(df)`: devuelve las filas no dominadas ordenadas por `bytes_vec` creciente (recorre en orden de memoria y conserva una fila solo si su `ndcg10` supera estrictamente el mejor visto).
- `recomienda(df, presupuesto_gb, n_docs)`: entre las filas que caben en el presupuesto (`bytes_vec * n_docs / 1e9 <= presupuesto_gb`), elige la de mayor `ndcg10` (desempate: menos bytes, luego menos `ms_query`).
Después dibuja la frontera (`bytes_vec` en escala log vs `ndcg10`) coloreando por precisión.

In [ ]:
def frontera_pareto(df):
    # TODO 7.a: filas no dominadas (menos memoria y mejor ndcg10), ordenadas por bytes_vec
    raise NotImplementedError("TODO 7.a")

def recomienda(df, presupuesto_gb=PRESUPUESTO_GB, n_docs=N_DOCS_PROD):
    # TODO 7.b: mejor configuración que cabe en el presupuesto
    raise NotImplementedError("TODO 7.b")

def dibuja_frontera(df, frontera):
    # TODO 7.c: scatter de todas las filas (color por precision) y línea que une la frontera
    raise NotImplementedError("TODO 7.c")

try:
    FRONT = frontera_pareto(RESULTS)
    REC = recomienda(RESULTS)
    dibuja_frontera(RESULTS, FRONT)
    plt.show()
    print("Recomendación:", REC.to_dict())
except (NotImplementedError, AttributeError, TypeError) as e:
    FRONT = REC = None
    print("⏳ pendiente:", repr(e))

## 12 · Autocomprobación de la rúbrica
Esta celda evalúa R1–R8 sobre **tus** funciones y resultados. Debe imprimir `✅` en todo cuando termines (R9 es manual: escribe tu conclusión en la celda de abajo).

In [ ]:
def autograder():
    checks = {}
    try:
        X = trunc_renorm(np.random.default_rng(1).normal(size=(50, 40)).astype(np.float32), 24)
        checks["R1 trunc_renorm norma 1"] = bool(np.abs(np.linalg.norm(X, axis=1) - 1).max() < 1e-5)
    except Exception as e:
        checks["R1 trunc_renorm norma 1"] = False
    try:
        from sentence_transformers.util import quantize_embeddings
        E = np.random.default_rng(2).normal(size=(60, 64)).astype(np.float32)
        lo, hi = ajusta_rangos(E)
        ok_bin = bool((cuantiza_binario(E) == quantize_embeddings(E, "ubinary")).all())
        d = np.abs(cuantiza_int8(E, lo, hi).astype(int) - quantize_embeddings(E, "int8", ranges=np.stack([lo, hi])).astype(int)).max()
        checks["R2 binario == ubinary e int8 ≤ 1 nivel"] = ok_bin and d <= 1
    except Exception as e:
        checks["R2 binario == ubinary e int8 ≤ 1 nivel"] = False
    R = RESULTS
    if R is not None:
        piv = R.pivot_table(index=["modelo", "dim"], columns="precision", values=["ndcg10", "overlap10"])
        checks["R3 int8 ≥ float32 − 0.03 (nDCG, todas las filas)"] = bool((piv["ndcg10"]["int8"] >= piv["ndcg10"]["float32"] - 0.03).all())
        checks["R4 overlap int8 medio ≥ 0.85"] = bool(piv["overlap10"]["int8"].mean() >= 0.85)
        checks["R5 rescoring ≥ binario en ≥ 90 % de filas"] = bool((piv["overlap10"]["binario+rescore"] >= piv["overlap10"]["binario"] - 1e-9).mean() >= 0.90)
        okmem = all(r.bytes_vec == BYTES[r.precision](r.dim) for r in R.itertuples())
        checks["R6 memoria exacta (int8=m, binario=m/8)"] = bool(okmem)
    else:
        for k in ("R3 int8 ≥ float32 − 0.03 (nDCG, todas las filas)", "R4 overlap int8 medio ≥ 0.85",
                  "R5 rescoring ≥ binario en ≥ 90 % de filas", "R6 memoria exacta (int8=m, binario=m/8)"):
            checks[k] = False
    checks["R7 Pareto ≥ 3 puntos y ≥ 1 no-float32"] = bool(FRONT is not None and len(FRONT) >= 3 and (FRONT["precision"] != "float32").any())
    checks["R8 recomendación dentro del presupuesto"] = bool(REC is not None and REC["bytes_vec"] * N_DOCS_PROD / 1e9 <= PRESUPUESTO_GB)
    for k, v in checks.items():
        print("✅" if v else "❌", k)
    return checks

_ = autograder()

## 13 · Tu conclusión (R9)
Escribe aquí (≤ 10 líneas, con números de tu tabla): ¿qué configuración recomiendas para 10 M de documentos en 20 GB, por qué, y qué riesgos tiene (licencia, latencia, calidad en código, drift)?

> *(Edita esta celda.)*

## 14 · Retos extra (nivel experto)
1. **PCA en lugar de truncado** para los modelos sin Matryoshka: compara la curva nDCG vs dimensión con y sin PCA (aprende la proyección con el corpus).
2. **Product Quantization** (anticipo del módulo 14): comprime a 32–64 bytes/vector y colócala en la gráfica de Pareto.
3. **Bootstrap pareado** (lección, sección 6): añade intervalos de confianza al nDCG y marca qué diferencias son significativas.
4. **Tu embedder del módulo 11** como un modelo más: ¿le gana a `multilingual-e5-small` en código?
5. **Asimetría**: indexa documentos con un modelo grande y consulta con uno pequeño de la misma familia (solo si comparten espacio de embeddings; compruébalo midiendo, no lo asumas).

## 15 · Reflexión (conecta con producción y con el hilo conductor)
- ¿Qué cambia si el 80 % de las queries son en español y los documentos están mayoritariamente en inglés? ¿Qué métrica añadirías por idioma?
- El rescoring `int8` vive en disco: ¿qué latencia p99 añade leer `k·over` vectores por query? (módulo 14)
- ¿Cómo versionarías el índice para migrar de embedder sin downtime? (módulo 18)
- ¿Qué parte de tu comparativa se rompería con un corpus de 100 M de documentos?

---
# ⛔ SPOILER — intenta resolverlo primero
Debajo está la solución de referencia completa. Si ejecutas las celdas siguientes **sobrescribes** tus funciones con las de la solución.

In [ ]:
# ── SOLUCIÓN DE REFERENCIA: TODO 1 ──
def codifica_modelo(clave, e):
    t0 = time.perf_counter(); Ed = e.encode_docs(DOC_TXT); ms_doc = (time.perf_counter() - t0) / len(DOC_TXT) * 1e3
    t0 = time.perf_counter(); Eq = e.encode_queries(Q_TXT); ms_q = (time.perf_counter() - t0) / len(Q_TXT) * 1e3
    return dict(Eq=Eq, Ed=Ed, ms_doc=ms_doc, ms_q=ms_q, params_M=e.n_params / 1e6, real=e.is_real)

# ── TODO 2 ──
def trunc_renorm(E, m):
    X = E[:, :m]
    return (X / np.maximum(np.linalg.norm(X, axis=1, keepdims=True), 1e-9)).astype(np.float32)

# ── TODO 3 ──
def ajusta_rangos(E_calib):
    return E_calib.min(0), E_calib.max(0)

def cuantiza_int8(E, lo, hi):
    paso = np.maximum(hi - lo, 1e-9) / 255.0
    return np.clip(np.round((E - lo) / paso) - 128, -128, 127).astype(np.int8)

def descuantiza_int8(Q, lo, hi):
    paso = np.maximum(hi - lo, 1e-9) / 255.0
    return (Q.astype(np.float32) + 128) * paso + lo

# ── TODO 4 ──
def cuantiza_binario(E, umbral=0.0):
    return np.packbits(E > umbral, axis=1)

def hamming(qb, db, bloque=2048):
    out = np.empty((len(qb), len(db)), dtype=np.int32)
    for s in range(0, len(db), bloque):
        out[:, s:s + bloque] = POPCNT[qb[:, None, :] ^ db[None, s:s + bloque, :]].sum(-1)
    return out

# ── TODO 5 ──
def construye_store(Ed, m):
    X = trunc_renorm(Ed, m)
    lo, hi = ajusta_rangos(X)
    return {"m": m, "f32": X, "int8": cuantiza_int8(X, lo, hi), "lo": lo, "hi": hi, "bin": cuantiza_binario(X)}

def buscar(Eq, store, precision, k=10, over=4):
    Q = trunc_renorm(Eq, store["m"])
    if precision == "float32":
        return topk_scores(Q @ store["f32"].T, k)
    if precision == "int8":
        return topk_scores(Q @ descuantiza_int8(store["int8"], store["lo"], store["hi"]).T, k)
    H = hamming(np.packbits(Q > 0, axis=1), store["bin"])
    if precision == "binario":
        return topk_scores(-H.astype(np.float32), k)
    cand = topk_scores(-H.astype(np.float32), min(k * over, H.shape[1]))
    out = np.empty((len(Q), k), dtype=np.int64)
    for i in range(len(Q)):
        v = descuantiza_int8(store["int8"][cand[i]], store["lo"], store["hi"])
        out[i] = cand[i][np.argsort(-(v @ Q[i]))[:k]]
    return out

In [ ]:
# ── TODO 6: la rejilla ──
def evalua_rejilla(banco):
    filas = []
    for nombre, c in banco.items():
        Eq, Ed = c["Eq"], c["Ed"]
        E_big = aumenta(Ed)
        for m in dims_grid(Ed.shape[1]):
            st_small, st_big = construye_store(Ed, m), construye_store(E_big, m)
            ref_m = buscar(Eq, st_big, "float32")                               # top-10 exacto a la misma dimensión m
            for prec in BYTES:
                res = evaluate(run_desde_indices(buscar(Eq, st_small, prec)), QRELS)
                t0 = time.perf_counter(); idx = buscar(Eq, st_big, prec); ms = (time.perf_counter() - t0) / len(Eq) * 1e3
                b = BYTES[prec](m)
                filas.append(dict(modelo=nombre, real=c["real"], dim=m, precision=prec, ndcg10=res["ndcg"], recall10=res["recall"],
                                  overlap10=overlap_at_k(idx, ref_m), bytes_vec=b, MB_10M=b * N_DOCS_PROD / 1e6, ms_query=ms))
    return pd.DataFrame(filas)

# ── TODO 7: Pareto y recomendación ──
def frontera_pareto(df):
    d = df.sort_values(["bytes_vec", "ndcg10"], ascending=[True, False])
    mejor, keep = -1.0, []
    for r in d.itertuples():
        if r.ndcg10 > mejor + 1e-12:
            keep.append(r.Index); mejor = r.ndcg10
    return d.loc[keep]

def recomienda(df, presupuesto_gb=PRESUPUESTO_GB, n_docs=N_DOCS_PROD):
    ok = df[df["bytes_vec"] * n_docs / 1e9 <= presupuesto_gb]
    return ok.sort_values(["ndcg10", "bytes_vec", "ms_query"], ascending=[False, True, True]).iloc[0]

def dibuja_frontera(df, frontera):
    colores = {"float32": "#1F6FEB", "int8": "#2CA02C", "binario": "#F28E2B", "binario+rescore": "#9467BD"}
    fig, ax = plt.subplots(figsize=(8, 4.4))
    for prec, g in df.groupby("precision"):
        ax.scatter(g["bytes_vec"], g["ndcg10"], s=28, alpha=.55, color=colores[prec], label=prec)
    ax.plot(frontera["bytes_vec"], frontera["ndcg10"], "k--", lw=1.4, label="frontera de Pareto")
    ax.axvline(PRESUPUESTO_GB * 1e9 / N_DOCS_PROD, color="#6B7280", ls=":", label=f"presupuesto {PRESUPUESTO_GB} GB / 10 M")
    ax.set_xscale("log", base=2); ax.set_xlabel("bytes por vector en RAM (log)"); ax.set_ylabel("nDCG@10")
    ax.set_title(f"Calidad vs memoria · corpus {CORPUS['source']}"); ax.legend(fontsize=8); ax.grid(alpha=.3)
    plt.tight_layout()

BANCO = construye_banco()
RESULTS = evalua_rejilla(BANCO)
FRONT = frontera_pareto(RESULTS)
REC = recomienda(RESULTS)
dibuja_frontera(RESULTS, FRONT); plt.show()
display(FRONT.round(3))
print("Recomendación:", REC[["modelo", "dim", "precision", "ndcg10", "bytes_vec", "MB_10M"]].to_dict())

In [ ]:
_ = autograder()
lat = RESULTS.groupby("precision")[["ms_query", "overlap10"]].mean().round(3)
display(lat)

### Conclusión de referencia (ejemplo)
Con los números de la tabla anterior: `int8` da casi el mismo nDCG@10 que `float32` ocupando 4× menos; la binaria sin rescoring pierde calidad y el rescoring recupera la mayor parte; recortar dimensiones en modelos sin Matryoshka pierde más que recortar bits. Para 10 M de documentos y 20 GB, la configuración recomendada es la de mayor nDCG@10 que cabe (ver `REC`), y debe validarse con el golden set real y con IC antes de producción. Riesgos: licencia del modelo (p. ej. pesos no comerciales), latencia del rescoring en disco, y que el corpus sintético no representa la distribución real.